# 04 – Systematic Error Analysis

**Purpose:** Perform a deep diagnostic analysis of the errors made by our best baseline classifier (`Linear SVM`) on the Banking77 customer support test split.

This notebook covers:
1. Loading prediction and misclassification datasets.
2. Finding the top 100 mistakes.
3. Identifying per-class accuracies and identifying the hardest classes.
4. Showcasing longest and shortest failures.
5. Listing the most frequently confused intent pairs.
6. Rendering and displaying a confusion matrix heatmap zoomed into the top error classes.

## 0. Setup and Environment

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import IPython.display as display

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.error_analysis import ErrorAnalyzer
from src.utils.constants import OUTPUT_DIR

predictions_path = OUTPUT_DIR / "metrics" / "predictions.csv"
model_name = "linear_svm"

analyzer = ErrorAnalyzer(predictions_path, model_name)
summary = analyzer.run_analysis_pipeline()
print("Error analysis pipelines executed and outputs exported.")

## 1. Top 100 Mistakes

Let's load the saved misclassified examples CSV and show the first 10 mistakes.

In [ ]:
misclassified_df = pd.read_csv(OUTPUT_DIR / "metrics" / "misclassified.csv")
print(f"Total misclassified instances: {len(misclassified_df)}")
print("\nFirst 10 mistakes:")
misclassified_df[["text", "label_text", "pred_label_text"]].head(10)

## 2. Per-Class Accuracy Analysis

Let's load the class metrics table and view classes with the lowest accuracy (hardest intents to classify).

In [ ]:
class_metrics_df = pd.read_csv(OUTPUT_DIR / "metrics" / "class_metrics.csv")
print("Top 10 Hardest Class Intents (Lowest Accuracy):")
class_metrics_df.sort_values("accuracy", ascending=True).head(10)

## 3. Longest and Shortest Failures

Reviewing length-based failure modes: queries that are very long (often containing multiple clauses or context) vs. queries that are very short (containing sparse keywords).

In [ ]:
longest_fails = analyzer.get_longest_failures(5)
print("Longest Failed Queries:")
for idx, row in longest_fails.iterrows():
    print(f"- Text ({row['text_len_chars']} chars): {row['text']}")
    print(f"  True: {row['label_text']} | Pred: {row['pred_label_text']}\n")

In [ ]:
shortest_fails = analyzer.get_shortest_failures(5)
print("Shortest Failed Queries:")
for idx, row in shortest_fails.iterrows():
    print(f"- Text ({row['text_len_chars']} chars): {row['text']}")
    print(f"  True: {row['label_text']} | Pred: {row['pred_label_text']}\n")

## 4. Most Confused Intent Pairs

Identify which pairs of class labels are mistranslated the most.

In [ ]:
confused_df = analyzer.get_most_confused_intents(10)
print("Top Confused Intent Pairs:")
confused_df

## 5. Confusion Heatmap visualization

Display the rendered confusion matrix heatmap zoom-in plot.

In [ ]:
display.Image(filename=str(OUTPUT_DIR / "metrics" / "plots" / "confusion_heatmap.png"))